# Visualización de Datos — StreamView Analytics

## Evaluación Parcial N°1 (Encargo) y N°2 (Presentación) — ADY1104

**Rol del equipo:** Consultoría especializada en analítica y comunicación visual de datos
**Cliente:** StreamView Analytics (plataforma internacional de streaming)
**Fuentes de datos:** Catálogo de Películas y Series (`netflix_movies_detailed_up_to_2025.csv`, `netflix_tv_shows_detailed_up_to_2025.csv`), 2010–2025

---

Este notebook documenta de manera integral el desarrollo del encargo: desde la definición del problema de negocio hasta las visualizaciones interactivas, la narrativa de datos (*data storytelling*) y las recomendaciones finales para la organización.

**Estructura del notebook:**
1. Problema de negocio y audiencia
2. Carga y descripción de datos
3. Análisis exploratorio de datos (EDA)
4. Diseño e implementación de visualizaciones (Plotly)
5. Narrativa visual (Data Storytelling)
6. Evaluación crítica de la solución
7. Conclusiones y recomendaciones finales

> **Nota técnica:** coloca este notebook en la misma carpeta que los dos archivos CSV (o ajusta las rutas de `pd.read_csv` en la Sección 2 si los organizas dentro de una subcarpeta `data/`, siguiendo la estructura de proyecto sugerida en la pauta).
>
> **Dashboard complementario:** este notebook viene acompañado de `dashboard_streamview_analytics.py`, un dashboard interactivo en Streamlit (KPIs, filtros, navegación por pestañas) que satisface el requisito de la pauta de entregar y mostrar un dashboard, no solo proponerlo. Detalle completo en la Sección 6.

## 1. Problema de negocio y audiencia

### 1.1 Contexto organizacional

StreamView Analytics es una plataforma internacional de streaming que compite por la atención de audiencias en decenas de mercados simultáneamente. Su ventaja competitiva depende de tres palancas que la Dirección de Contenido monitorea de forma constante: **retención de suscriptores**, **nivel de interacción (engagement)** con el catálogo, y **acierto en las preferencias de consumo** de contenido a través de géneros, formatos y países.

Actualmente, las decisiones de inversión en contenido (qué producir, en qué género, en qué mercado, en qué formato) se apoyan en reportes dispersos y con poca visibilidad conjunta entre el catálogo de Películas y el de Series. La Dirección de Contenido contrata a nuestro equipo consultor externo para **construir una solución de analítica visual** que integre ambas fuentes y entregue una lectura clara, defendible y accionable del catálogo.

### 1.2 Problema de negocio

> ¿Qué segmentos del catálogo (formato, género, país de origen) generan mayor engagement y mejor percepción de calidad por parte de la audiencia, y cómo debería StreamView Analytics priorizar su inversión de contenido para maximizar retención y satisfacción de usuarios?

### 1.3 Audiencia objetivo

La solución está diseñada para el **Comité Directivo de Contenido de StreamView Analytics** (Chief Content Officer, VP de Estrategia de Contenido y Dirección General), con dos audiencias secundarias que también usarán este material: el equipo de **Data & Analytics** (que reutilizará el notebook como base técnica) y el equipo de **Marketing / Comunicación de audiencias**.

Esto impone un requisito de diseño clave: **el mensaje debe entenderse en segundos**, sin requerir conocimientos de estadística ni de las herramientas utilizadas. Los ejecutivos no leen tablas de coeficientes; leen titulares, comparan barras y confían en un color cuando ya saben lo que significa.

### 1.4 Propósito comunicacional

Comunicar, con evidencia verificable, **en qué segmentos del catálogo conviene concentrar la inversión de contenido** — y en cuáles conviene ser cauteloso — a través de una narrativa visual breve, interactiva y jerarquizada, capaz de resistir preguntas cruzadas en una defensa técnica en vivo.

### 1.5 Estrategia de comunicación

Se optó deliberadamente por **visualizaciones interactivas en Plotly** (en lugar de gráficos estáticos) por tres razones ligadas directamente a la audiencia y al formato de entrega:

1. **La evaluación exige defensa oral con preguntas cruzadas.** La interactividad (*hover*, zoom, leyenda clickeable) permite explorar el dato en vivo frente al comité, sin tener que preparar decenas de gráficos estáticos adicionales para cada posible pregunta.
2. **Jerarquía de información bajo demanda.** Cada gráfico expone un mensaje principal a simple vista (título comunicacional + codificación visual mínima) y reserva el detalle granular (título exacto, país, cifras precisas) para el *hover*, evitando saturar el gráfico y respetando el principio de minimizar la carga cognitiva.
3. **Coherencia con el estándar de la industria.** Herramientas como Plotly/Dash/Streamlit son las que efectivamente usan los equipos de analítica de plataformas de streaming para *reporting* ejecutivo. De hecho, esta misma lógica de análisis se entrega también como **dashboard interactivo complementario** (`dashboard_streamview_analytics.py`, ver Sección 6) con KPIs, filtros y navegación en vivo — este notebook es el informe de análisis; el dashboard es la herramienta de exploración para el Comité.

## 2. Carga y descripción de datos

Se integran dos fuentes de datos corporativas provistas por StreamView Analytics:

| Fuente | Descripción | Filas | Columnas |
|---|---|---|---|
| `netflix_movies_detailed_up_to_2025.csv` | Catálogo de Películas, 2010–2025 | 16.000 | 18 |
| `netflix_tv_shows_detailed_up_to_2025.csv` | Catálogo de Series TV, 2010–2025 | 16.000 | 16 |

A continuación se cargan ambas fuentes y se documenta su estructura antes de cualquier limpieza, tal como corresponde a una auditoría de datos rigurosa.

In [ ]:
# Librerías necesarias. Si tu entorno no las tiene instaladas, ejecuta antes en una celda:
# !pip install pandas plotly

import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print("Librerías cargadas correctamente.")

In [ ]:
# Carga de las dos fuentes de datos corporativas
df_movies_raw = pd.read_csv('netflix_movies_detailed_up_to_2025.csv')
df_tv_raw = pd.read_csv('netflix_tv_shows_detailed_up_to_2025.csv')

print(f"Películas: {df_movies_raw.shape[0]:,} filas x {df_movies_raw.shape[1]} columnas")
print(f"Series TV: {df_tv_raw.shape[0]:,} filas x {df_tv_raw.shape[1]} columnas")

In [ ]:
print("=== Estructura — Películas ===")
df_movies_raw.info()
print("\n=== Estructura — Series TV ===")
df_tv_raw.info()

### 2.1 Descripción de variables relevantes

| Variable | Tipo | Presente en | Descripción |
|---|---|---|---|
| `show_id` | Identificador | Ambas | Identificador numérico del título (namespace independiente entre Películas y Series) |
| `title` | Categórica (texto) | Ambas | Nombre del título |
| `director`, `cast` | Categórica (texto) | Ambas | Ficha técnica y de reparto |
| `country` | Categórica multivaluada | Ambas | País(es) de producción, separados por coma |
| `genres` | Categórica multivaluada | Ambas | Género(s) del título, separados por coma |
| `language` | Categórica (código ISO 639-1) | Ambas | Idioma original |
| `release_year` | Temporal | Ambas | Año de lanzamiento (2010–2025) |
| `date_added` | Temporal | Ambas | Fecha de incorporación al catálogo *(ver hallazgo de calidad de datos, Sección 2.3)* |
| `popularity` | Numérica continua | Ambas | Índice de popularidad/buzz reciente (proxy de **engagement**) |
| `vote_count` | Numérica discreta | Ambas | N.º de votos de audiencia acumulados (proxy de **alcance**) |
| `vote_average` | Numérica continua (0–10) | Ambas | Calificación promedio de audiencia (proxy de **percepción de calidad**) |
| `rating` | Numérica continua | Ambas | *(ver hallazgo de calidad de datos: idéntica a `vote_average`)* |
| `duration` | — | Ambas | *(ver hallazgo de calidad de datos: no utilizable)* |
| `budget`, `revenue` | Numérica continua | Solo Películas | Presupuesto e ingresos reportados (alta proporción de datos en cero) |

Las variables centrales para responder el problema de negocio son **`content_type`** (a construir), **`genres`**, **`country`**, **`release_year`**, **`popularity`**, **`vote_count`** y **`vote_average`**.

In [ ]:
# Auditoría rápida de budget/revenue (solo Películas) antes de decidir si se incluyen en el análisis
pct_budget_0 = (df_movies_raw['budget'] == 0).mean() * 100
pct_revenue_0 = (df_movies_raw['revenue'] == 0).mean() * 100

print(f"% de películas con budget = 0 (no reportado): {pct_budget_0:.1f}%")
print(f"% de películas con revenue = 0 (no reportado): {pct_revenue_0:.1f}%")
print("→ Estas columnas tampoco existen en el catálogo de Series.")
print("  Decisión: se excluyen del análisis visual principal (justificación completa en la Sección 6).")

### 2.2 Limpieza e integración

Antes de integrar ambas fuentes se realizan las siguientes acciones, cada una justificada por un hallazgo verificado en los datos:

| Acción | Justificación (hallazgo verificado en el código) |
|---|---|
| Eliminar duplicados por `show_id` | Se detectaron 9 filas duplicadas en el catálogo de Series |
| Eliminar columna `duration` | 100% nula en Películas; constante `"1 Seasons"` en el 100% de las Series → no aporta información |
| Eliminar columna `rating` | Idéntica a `vote_average` en el 100% de las filas (columna redundante) |
| Crear bandera `sin_calificacion` | `vote_average = 0` coincide con `vote_count ≈ 0` en la enorme mayoría de los casos → representa **ausencia de datos**, no una calificación real de 0/10 |
| Crear `content_type` | Para poder comparar Películas vs. Series tras la integración |
| Crear `id_unico` | `show_id` tiene namespaces independientes entre Películas y Series (cientos de valores coinciden numéricamente sin ser el mismo título) |

In [ ]:
def limpiar_dataset(df, tipo_contenido):
    """Limpia y estandariza un dataset de StreamView Analytics (Películas o Series)."""
    d = df.copy()

    # 1) Duplicados
    d = d.drop_duplicates()
    d = d.drop_duplicates(subset=['show_id'], keep='first')

    # 2) Columnas no utilizables (ver justificación en la celda de markdown anterior)
    columnas_a_eliminar = [c for c in ['duration', 'rating'] if c in d.columns]
    d = d.drop(columns=columnas_a_eliminar)

    # 3) Limpieza de texto (espacios en blanco)
    for col in ['title', 'director', 'cast', 'country', 'genres', 'language', 'description']:
        if col in d.columns:
            d[col] = d[col].astype('string').str.strip()

    # 4) Variables derivadas
    d['release_year'] = d['release_year'].astype(int)
    d['content_type'] = tipo_contenido
    d['sin_calificacion'] = d['vote_average'] == 0
    d['id_unico'] = tipo_contenido[:3].upper() + '_' + d['show_id'].astype(str)

    return d


df_movies = limpiar_dataset(df_movies_raw, 'Película')
df_tv = limpiar_dataset(df_tv_raw, 'Serie TV')

print(f"Películas tras limpieza: {df_movies.shape[0]:,} filas")
print(f"Series TV tras limpieza: {df_tv.shape[0]:,} filas")

In [ ]:
# Integración: columnas comunes a ambas fuentes
columnas_comunes = [
    'id_unico', 'show_id', 'content_type', 'title', 'director', 'cast', 'country',
    'release_year', 'genres', 'language', 'description',
    'popularity', 'vote_count', 'vote_average', 'sin_calificacion'
]

df_catalogo = pd.concat([df_movies[columnas_comunes], df_tv[columnas_comunes]], ignore_index=True)

print(f"Catálogo integrado: {df_catalogo.shape[0]:,} títulos "
      f"({(df_catalogo.content_type=='Película').sum():,} películas + "
      f"{(df_catalogo.content_type=='Serie TV').sum():,} series)")

df_catalogo.head(3)

In [ ]:
def explode_columna(df, columna):
    """Convierte una columna multivaluada (separada por coma) en una fila por valor."""
    d = df.assign(**{columna: df[columna].fillna('').str.split(', ')}).explode(columna)
    d[columna] = d[columna].str.strip()
    return d[d[columna] != '']


print("=== Valores nulos en el catálogo integrado ===")
print(df_catalogo.isnull().sum())

print("\n=== Títulos sin datos de calificación de audiencia (vote_average = 0) ===")
print(df_catalogo.groupby('content_type')['sin_calificacion'].sum())

### 2.3 Hallazgos de calidad de datos detectados durante la carga

Antes de avanzar al análisis exploratorio, documentamos tres hallazgos que condicionan el resto del proyecto (y que se retoman en la Sección 6, Evaluación Crítica):

1. **`date_added` no aporta información independiente de `release_year`.** Se verificó que el año de ambas columnas coincide en el 100% de los títulos, en ambas fuentes. Es decir, no podemos distinguir "cuándo se sumó al catálogo de streaming" de "cuándo se estrenó" — por lo tanto, el análisis temporal de este proyecto usa exclusivamente `release_year`.

2. **El catálogo original contiene exactamente 1.000 títulos por año, cada año, entre 2010 y 2025, en cada fuente.** Esta uniformidad perfecta indica que se trata de una **muestra curada/estratificada**, no del historial real de crecimiento del catálogo (se verifica con los datos originales en la Sección 3.1). En consecuencia, este proyecto **evita construir un gráfico de "volumen de contenido a través del tiempo"**, ya que sería plano por diseño muestral y no reflejaría una tendencia de negocio real.

3. **Los títulos más recientes (2024–2025) muestran un sesgo de "arranque en frío" (*cold start*) en `vote_count` y `popularity`**, no así en `vote_average`. Esto es esperable: un título recién lanzado aún no acumula votos ni tiempo de exposición, pero su calificación promedio (cuando ya tiene votos) es igual de confiable que la de un título antiguo. Se verifica en la Sección 3.2 y se maneja aplicando umbrales mínimos de `vote_count` en los gráficos que comparan popularidad entre títulos.

## 3. Análisis exploratorio de datos (EDA)

In [ ]:
print("=== Estadística descriptiva — variables numéricas (catálogo integrado) ===")
df_catalogo[['popularity', 'vote_count', 'vote_average']].describe()

Las tres métricas numéricas centrales muestran distribuciones fuertemente asimétricas: `popularity` y `vote_count` tienen colas largas hacia la derecha (pocos títulos extremadamente populares elevan el promedio muy por encima de la mediana), mientras que `vote_average` se comporta de forma mucho más simétrica alrededor de 6–7 puntos. Esta asimetría es la razón por la que, más adelante, usamos **escala logarítmica** para `popularity` en el gráfico de dispersión (Gráfico 3): en escala lineal, un puñado de títulos "virales" comprimiría visualmente a todos los demás contra el eje.

### 3.1 Verificación del patrón de muestreo por año

In [ ]:
# Se usan los datos ORIGINALES (antes de limpieza) para caracterizar el diseño muestral de la fuente,
# sin que la eliminación de duplicados distorsione el patrón real de la fuente de datos.
print("=== Títulos por año de lanzamiento (datos originales) ===")
comparacion_anual = pd.DataFrame({
    'Películas': df_movies_raw['release_year'].value_counts().sort_index(),
    'Series TV': df_tv_raw['release_year'].value_counts().sort_index()
})
comparacion_anual

Se confirma el hallazgo de la Sección 2.3: **exactamente 1.000 títulos por año en cada fuente**, sin variación, entre 2010 y 2025. Por diseño, cualquier gráfico de "cantidad de títulos por año" sería una línea recta — no un hallazgo de negocio. Este es un ejemplo concreto de por qué la auditoría de datos debe preceder al diseño de visualizaciones: graficar esto sin revisarlo antes habría producido un gráfico correcto técnicamente, pero **comunicacionalmente engañoso**.

### 3.2 Verificación del sesgo de "arranque en frío" en títulos recientes

In [ ]:
print("=== Vote count y popularidad promedio en los años más recientes ===")
patron_temporal = df_catalogo.groupby(['content_type', 'release_year']).agg(
    vote_count_promedio=('vote_count', 'mean'),
    popularidad_promedio=('popularity', 'mean')
).round(1)

patron_temporal.loc[(slice(None), [2019, 2020, 2021, 2022, 2023, 2024, 2025]), :]

El `vote_count` promedio cae de forma pronunciada en los últimos años (en Películas pasa de ~870 votos en 2019 a ~11 en 2025), mientras que — como se muestra en el Gráfico 2 — la calificación promedio **no** cae de la misma forma. Esto confirma que la caída es un artefacto del **tiempo de acumulación de votos**, no una señal real de que el contenido reciente sea peor recibido. Por eso, en el Gráfico 3 filtramos por un mínimo de votos acumulados: comparar un título de 2025 con pocos votos contra uno de 2015 con miles de votos no sería una comparación justa.

### 3.3 Distribución por género

In [ ]:
print("=== Top 10 géneros — Películas ===")
top_generos_pelis = explode_columna(df_movies, 'genres')['genres'].value_counts().head(10)
print(top_generos_pelis)

print("\n=== Top 10 géneros — Series TV ===")
top_generos_series = explode_columna(df_tv, 'genres')['genres'].value_counts().head(10)
print(top_generos_series)

Drama y Comedia lideran en ambos formatos, lo cual era esperable. Más interesante: **las taxonomías de género de TMDB difieren entre Películas y Series** (por ejemplo, "Action" y "Science Fiction" en Películas se agrupan como "Action & Adventure" y "Sci-Fi & Fantasy" en Series). Solo **8 etiquetas de género son textualmente idénticas entre ambas fuentes** (Drama, Comedy, Animation, Crime, Family, Mystery, Documentary, Western); el Gráfico 1 se construye exclusivamente sobre esas 8 categorías para garantizar una comparación Película-vs-Serie válida, sin forzar equivalencias entre taxonomías distintas.

### 3.4 Distribución por país

In [ ]:
print("=== Top 10 países — Películas ===")
top_paises_pelis = explode_columna(df_movies, 'country')['country'].value_counts().head(10)
print(top_paises_pelis)

print("\n=== Top 10 países — Series TV ===")
top_paises_series = explode_columna(df_tv, 'country')['country'].value_counts().head(10)
print(top_paises_series)

Estados Unidos domina el volumen en ambos catálogos, pero el patrón internacional difiere marcadamente por formato: en Series TV, **China y Japón ocupan el 2° y 3er lugar** en volumen (superando a Reino Unido y Canadá), mientras que en Películas esos mismos países aparecen mucho más abajo en el ranking. Esto sugiere que la estrategia de contenido internacional de StreamView Analytics está más desarrollada en Series que en Películas — patrón que se profundiza en el Gráfico 4.

### 3.5 Diversidad de idiomas

In [ ]:
print("=== Diversidad de idiomas en el catálogo integrado ===")
idiomas = df_catalogo['language'].value_counts()
top_idiomas = idiomas.head(8)
pct_ingles = idiomas.get('en', 0) / len(df_catalogo) * 100
pct_asia_oriental = idiomas.reindex(['zh', 'ja', 'ko']).sum() / len(df_catalogo) * 100

print(f"Idiomas distintos en el catálogo: {df_catalogo['language'].nunique()}")
print(f"% del catálogo en inglés: {pct_ingles:.1f}%")
print(f"% del catálogo en chino + japonés + coreano (combinado): {pct_asia_oriental:.1f}%")
print("\nTop 8 idiomas:")
print(top_idiomas)

El catálogo abarca **más de 80 idiomas distintos**. Aunque el inglés predomina, cerca de una cuarta parte del catálogo está en idiomas de Asia Oriental (chino, japonés, coreano combinados) — consistente con el peso de China, Japón y Corea del Sur observado en el análisis por país.

### Síntesis del EDA

Antes de diseñar las visualizaciones, consolidamos los patrones que orientarán el diseño:

- Las **Series TV superan a las Películas** en calificación promedio y en popularidad mediana (se cuantifica en el Gráfico 2 y en la síntesis de la Sección 5).
- La comparación de género válida entre formatos se limita a **8 categorías textualmente comparables**.
- **Japón y China** combinan alto volumen y alta calificación en Series — un patrón que amerita un gráfico dedicado.
- `popularity` y `vote_average` deben examinarse juntas con cautela: aún no sabemos si están correlacionadas (se resuelve en el Gráfico 3).
- Cualquier comparación de popularidad entre años debe filtrar por `vote_count` mínimo para evitar el sesgo de arranque en frío.

Con esta base, pasamos al diseño de las visualizaciones.

## 4. Diseño e implementación de visualizaciones (Plotly)

Se construyen 5 visualizaciones interactivas con `plotly.express`, cada una seleccionada para responder una pregunta de negocio específica y cada una acompañada de su justificación teórica (tipo de dato, atributos preatentivos, Gestalt y carga cognitiva).

Toda la sección utiliza una **paleta corporativa única y sobria**, definida una sola vez y reutilizada en los 5 gráficos para garantizar coherencia visual (principio de **similitud**: el mismo color siempre significa lo mismo en todo el documento).

In [ ]:
# Paleta corporativa StreamView Analytics.
# Principio de diseño: pocos colores, cada uno con un significado fijo y consistente
# en las 5 visualizaciones (evita que la audiencia deba "reaprender" el color en cada gráfico).
PALETA = {
    'pelicula': '#C97B30',  # ámbar/terracota — SIEMPRE representa "Película"
    'serie':    '#0B3D62',  # azul marino     — SIEMPRE representa "Serie TV"
    'neutral':  '#B0B7C3',  # gris            — líneas de referencia / contexto
    'acento':   '#E4572E',  # coral           — reservado para resaltar UN hallazgo puntual
    'texto':    '#1F2937',
}
ESCALA_SECUENCIAL = ['#EAF0F7', '#7FA1C4', '#0B3D62']  # para codificar variables continuas (un solo matiz)

colores_tipo = {'Película': PALETA['pelicula'], 'Serie TV': PALETA['serie']}

# Configuración común a los 5 gráficos (tipografía, márgenes, plantilla)
LAYOUT_BASE = dict(
    template='plotly_white',
    font=dict(family='Arial, sans-serif', size=13, color=PALETA['texto']),
    title_font=dict(size=17),
    margin=dict(l=10, r=10, t=70, b=10),
)

print("Paleta corporativa definida.")

### 4.1 Gráfico 1 — Volumen de catálogo por género (Película vs. Serie)

**Pregunta de negocio:** de los géneros donde ambos formatos usan la misma clasificación, ¿en cuáles ya invierte más StreamView Analytics en Series que en Películas?

In [ ]:
generos_comparables = ['Drama', 'Comedy', 'Animation', 'Crime', 'Family', 'Mystery', 'Documentary', 'Western']

datos_g1 = explode_columna(df_catalogo, 'genres')
datos_g1 = datos_g1[datos_g1['genres'].isin(generos_comparables)]

chart1_df = datos_g1.groupby(['genres', 'content_type']).size().reset_index(name='titulos')

# Orden ascendente por volumen total: en un gráfico de barras horizontal,
# Plotly dibuja el último elemento de la lista en la parte SUPERIOR.
orden_generos = (chart1_df.groupby('genres')['titulos'].sum()
                  .sort_values(ascending=True).index.tolist())

fig1 = px.bar(
    chart1_df,
    x='titulos', y='genres', color='content_type',
    orientation='h', barmode='group',
    color_discrete_map=colores_tipo,
    category_orders={'genres': orden_generos, 'content_type': ['Película', 'Serie TV']},
    labels={'titulos': 'Número de títulos', 'genres': '', 'content_type': ''},
    title='En volumen, las Series ya superan a las Películas en la mayoría de los géneros comparables',
    text='titulos',
)
fig1.update_traces(texttemplate='%{text:,}', textposition='outside', cliponaxis=False)
fig1.update_layout(
    **LAYOUT_BASE,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    xaxis=dict(showgrid=True, gridcolor='#E5E7EB'),
    yaxis=dict(title=''),
    height=480,
    bargap=0.25,
)
fig1.show()

**Justificación teórica — Gráfico 1**

- **Selección según tipo de dato:** comparamos una variable categórica (género) segmentada por otra categórica (tipo de contenido) frente a un conteo. Las **barras** son la representación estándar para comparar magnitudes entre categorías discretas — a diferencia de líneas (pensadas para tendencias continuas) o dispersión (pensada para relaciones entre variables numéricas). Se usa orientación horizontal porque las etiquetas de género son largas y así se leen sin rotar texto.
- **Atributos visuales y preatentivos:** la **longitud** de la barra (uno de los atributos preatentivos de mayor precisión perceptual) codifica el volumen — la comparación más importante. El **color** se reserva exclusivamente para diferenciar Película/Serie (2 categorías, bajo esfuerzo de decodificación), evitando introducir una tercera variable por color que saturaría el gráfico. La **posición** (orden de mayor a menor volumen) permite un barrido visual único de arriba hacia abajo sin necesidad de leer las etiquetas numéricas.
- **Percepción visual y Gestalt:** las barras de cada género están **agrupadas por proximidad** (Película y Serie contiguas dentro de cada categoría), lo que activa el principio de *proximidad* para que el ojo las lea como un par comparable. El uso del mismo par de colores en todo el documento aplica el principio de *similitud* — el lector no necesita releer la leyenda en cada gráfico. El espacio en blanco entre grupos de género separa categorías sin necesidad de líneas divisorias.
- **Carga cognitiva:** se eliminan líneas de grilla verticales innecesarias, se muestran las etiquetas de valor directamente sobre cada barra (`texttemplate`) para que el usuario no tenga que interpolar contra el eje, y el título comunica el hallazgo (no solo el contenido del gráfico), reduciendo el esfuerzo de interpretación antes de mirar el detalle.

### 4.2 Gráfico 2 — Evolución de la calificación promedio por año de lanzamiento

**Pregunta de negocio:** ¿la ventaja de calidad percibida de las Series sobre las Películas es un fenómeno reciente, o es estructural?

In [ ]:
chart2_df = (df_catalogo[~df_catalogo['sin_calificacion']]
             .groupby(['release_year', 'content_type'])['vote_average']
             .mean().reset_index()
             .rename(columns={'vote_average': 'calificacion_promedio'}))

fig2 = px.line(
    chart2_df,
    x='release_year', y='calificacion_promedio', color='content_type',
    color_discrete_map=colores_tipo, markers=True,
    category_orders={'content_type': ['Película', 'Serie TV']},
    labels={'release_year': 'Año de lanzamiento',
            'calificacion_promedio': 'Calificación promedio (0–10)', 'content_type': ''},
    title='La brecha de calificación entre Series y Películas se mantiene estable durante 16 años',
)
fig2.update_traces(line=dict(width=3), marker=dict(size=7))
fig2.add_vrect(
    x0=2024.5, x1=2025.5, fillcolor=PALETA['neutral'], opacity=0.15, line_width=0,
    annotation_text='2025: pocos votos<br>acumulados aún', annotation_position='top left',
    annotation_font_size=10,
)
fig2.update_layout(
    **LAYOUT_BASE,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    xaxis=dict(dtick=2, showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='#E5E7EB', range=[5.5, 8]),
    height=440,
)
fig2.show()

**Justificación teórica — Gráfico 2**

- **Selección según tipo de dato:** `release_year` es una variable **temporal continua** y `calificacion_promedio` es numérica — el caso de uso canónico de un **gráfico de líneas**, que representa evolución en el tiempo mucho mejor que barras (que enfatizarían valores puntuales, no la tendencia) o puntos sueltos (que no comunican dirección).
- **Atributos visuales y preatentivos:** se usa **posición vertical** (altura de cada punto) para codificar el valor exacto y **color** para separar las dos series de tiempo — nuevamente los mismos dos colores del Gráfico 1 (principio de similitud). El grosor de línea y el tamaño de marcador se aumentaron levemente respecto al valor por defecto para que la comparación de pendientes sea legible incluso proyectada en una sala (defensa oral).
- **Percepción visual y Gestalt:** el principio de **conexión** es el que hace funcionar esta visualización: los puntos discretos de cada año se perciben como una sola entidad (una tendencia) porque están unidos por una línea continua del mismo color — sin esa conexión tendríamos una nube de puntos difícil de leer como evolución. La franja sombreada en 2025 usa *proximidad* y contraste de opacidad para señalar "este tramo es distinto" sin necesitar una leyenda adicional.
- **Carga cognitiva:** el eje Y no parte de 0 — a diferencia de un gráfico de barras (donde truncar el eje distorsiona la percepción de magnitud por longitud), en un gráfico de líneas la comparación es de **posición y pendiente**, por lo que acotar el rango (5,5–8) amplía visualmente la brecha real sin introducir un engaño perceptual, evitando "aplanar" una diferencia que sí importa para la decisión de negocio. La anotación de 2025 anticipa la pregunta más probable de la audiencia ("¿por qué cae el último punto?") directamente en el gráfico, evitando que se lea como una caída real de calidad.

### 4.3 Gráfico 3 — Relación entre popularidad y calificación de audiencia

**Pregunta de negocio:** si un título es muy popular, ¿podemos asumir que también está bien evaluado? ¿Conviene usar una sola métrica para decidir qué producir?

In [ ]:
chart3_df = df_catalogo[
    (~df_catalogo['sin_calificacion']) & (df_catalogo['vote_count'] >= 100)
].copy()

correlacion = chart3_df[['popularity', 'vote_average']].corr().iloc[0, 1]
print(f"Correlación popularidad vs. calificación (títulos con ≥100 votos): r = {correlacion:.2f}")

fig3 = px.scatter(
    chart3_df,
    x='popularity', y='vote_average', color='content_type',
    color_discrete_map=colores_tipo, log_x=True, opacity=0.35,
    render_mode='webgl',
    hover_name='title',
    hover_data={'content_type': True, 'vote_count': ':,', 'popularity': ':.1f', 'vote_average': ':.2f'},
    category_orders={'content_type': ['Película', 'Serie TV']},
    labels={'popularity': 'Popularidad (escala logarítmica)',
            'vote_average': 'Calificación de audiencia (0–10)', 'content_type': ''},
    title='Popularidad y calificación son señales casi independientes: el alcance no garantiza calidad',
)
fig3.update_traces(marker=dict(size=5, line=dict(width=0)))
fig3.update_layout(
    **LAYOUT_BASE,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    xaxis=dict(showgrid=True, gridcolor='#E5E7EB'),
    yaxis=dict(showgrid=True, gridcolor='#E5E7EB', range=[0, 10.3]),
    height=520,
)
fig3.show()

**Justificación teórica — Gráfico 3**

- **Selección según tipo de dato:** ambas variables (`popularity`, `vote_average`) son **numéricas continuas** — el caso de uso exacto de un **gráfico de dispersión**, la única representación que permite evaluar visualmente la relación (o ausencia de relación) entre dos magnitudes a nivel de título individual. Se filtra a títulos con ≥100 votos para no mezclar la relación real con el ruido del sesgo de "arranque en frío" identificado en la Sección 3.2.
- **Atributos visuales y preatentivos:** la **posición** (x, y) de cada punto es el atributo preatentivo principal. El **color** vuelve a codificar únicamente el tipo de contenido (2 categorías, coherente con los gráficos anteriores). Se usa **opacidad reducida (35%)** en lugar de agregar un tercer atributo visual: con miles de puntos, la opacidad permite que las zonas de alta densidad se "iluminen" por superposición, funcionando como un mapa de calor implícito sin añadir una variable visual adicional que compita por atención.
- **Percepción visual y Gestalt:** la **similitud** de color agrupa perceptualmente los puntos de Serie vs. Película incluso sin fronteras explícitas, permitiendo detectar si un tipo de contenido ocupa una región distinta del plano. El espacio en blanco (fondo `plotly_white`, sin grilla de fondo saturada) evita que las líneas de referencia compitan visualmente con los propios puntos de datos, que son el verdadero protagonista.
- **Carga cognitiva:** se usa **escala logarítmica en X** precisamente para reducir carga cognitiva: en escala lineal, la fuerte asimetría de `popularity` (ver Sección 3) comprimiría el 90% de los títulos contra el eje izquierdo, obligando a la audiencia a "adivinar" patrones invisibles. El título comunica directamente la conclusión estadística (correlación débil) en lenguaje ejecutivo, evitando que el comité tenga que interpretar la nube de puntos por sí mismo para llegar al mismo hallazgo.

### 4.4 Gráfico 4 — Top 10 países productores de Series TV: volumen y calificación

**Pregunta de negocio:** ¿en qué mercados internacionales conviene reforzar la producción de series originales?

In [ ]:
tv_calificadas = df_tv[df_tv['vote_average'] > 0]

vol_paises = explode_columna(df_tv, 'country').groupby('country').size().reset_index(name='titulos')
rating_paises = (explode_columna(tv_calificadas, 'country')
                  .groupby('country')['vote_average'].mean()
                  .reset_index(name='calificacion_promedio'))

chart4_df = (vol_paises.merge(rating_paises, on='country', how='left')
             .sort_values('titulos', ascending=True)
             .tail(10))  # top 10 por volumen, en orden ascendente (el mayor queda arriba en el gráfico)

fig4 = px.bar(
    chart4_df,
    x='titulos', y='country', color='calificacion_promedio',
    color_continuous_scale=ESCALA_SECUENCIAL,
    orientation='h', text='titulos',
    category_orders={'country': chart4_df['country'].tolist()},
    labels={'titulos': 'Número de series en el catálogo', 'country': '',
            'calificacion_promedio': 'Calificación<br>promedio'},
    title='Japón y China: el mayor volumen de series internacionales también tiene alta calificación',
)
fig4.update_traces(texttemplate='%{text:,}', textposition='outside', cliponaxis=False)
fig4.update_layout(
    **LAYOUT_BASE,
    xaxis=dict(showgrid=True, gridcolor='#E5E7EB'),
    yaxis=dict(title=''),
    coloraxis_colorbar=dict(title='Calificación<br>promedio', tickformat='.1f'),
    height=480,
)
fig4.show()

**Justificación teórica — Gráfico 4**

- **Selección según tipo de dato:** `country` es categórica (nominal) y `titulos` es un conteo — nuevamente barras, coherente con el Gráfico 1. La diferencia es que aquí se **suma una segunda variable numérica continua** (`calificacion_promedio`) sin cambiar de tipo de gráfico, mostrando que un mismo formato puede escalar en información sin escalar en complejidad de lectura.
- **Atributos visuales y preatentivos:** la **longitud** de la barra sigue codificando volumen (la variable más importante para "dónde ya existe escala"). La **segunda variable (calificación) se codifica con color en escala secuencial de un solo matiz** (de gris-azulado claro a azul marino oscuro) — un atributo preatentivo distinto (intensidad/saturación) que se percibe simultáneamente con la longitud sin competir por atención, permitiendo leer "volumen Y calidad" en una sola mirada.
- **Percepción visual y Gestalt:** se usa una escala de **un solo matiz** (no arcoíris) deliberadamente: con un solo matiz, el cerebro interpreta la progresión de color como una escala ordinal natural (más oscuro = más alto), aplicando el principio de *similitud* de forma monotónica. Una escala multicolor (por ejemplo, semáforo rojo-amarillo-verde) introduciría categorías falsas donde en realidad solo hay un continuo.
- **Carga cognitiva:** limitar el gráfico a los **10 países de mayor volumen** (en vez de mostrar los más de 100 países presentes en los datos) es en sí mismo una decisión de reducción de carga cognitiva — el resto tiene peso testimonial para la conversación de inversión de contenido. El título nombra directamente los dos países que combinan ambas señales positivas, ahorrándole a la audiencia el trabajo de cruzar mentalmente longitud y color.

### 4.5 Gráfico 5 — Portafolio de géneros: popularidad vs. calificación (con volumen)

**Pregunta de negocio:** dentro de nuestro portafolio de géneros, ¿cuáles son apuestas seguras, cuáles son joyas de nicho, y cuáles son un riesgo a revisar?

In [ ]:
base_g5 = df_catalogo[df_catalogo['vote_count'] >= 50].copy()
exploded_g5 = explode_columna(base_g5, 'genres')

chart5_df = exploded_g5.groupby(['content_type', 'genres']).agg(
    titulos=('title', 'count'),
    calificacion_promedio=('vote_average', 'mean'),
    popularidad_promedio=('popularity', 'mean'),
).reset_index()
chart5_df = chart5_df[chart5_df['titulos'] >= 150]  # se descartan combinaciones con poca evidencia

fig5 = px.scatter(
    chart5_df,
    x='popularidad_promedio', y='calificacion_promedio', size='titulos',
    color='content_type', color_discrete_map=colores_tipo,
    hover_name='genres', size_max=42,
    category_orders={'content_type': ['Película', 'Serie TV']},
    labels={'popularidad_promedio': 'Popularidad promedio del género',
            'calificacion_promedio': 'Calificación promedio (0–10)',
            'content_type': '', 'titulos': 'N.º de títulos'},
    title='El género Documental es el más valorado en Películas, pero también el menos popular',
)
fig5.update_traces(marker=dict(line=dict(width=1, color='white')))

# Línea de referencia: promedio general del catálogo calificado
promedio_general = df_catalogo.loc[~df_catalogo['sin_calificacion'], 'vote_average'].mean()
fig5.add_hline(
    y=promedio_general, line_dash='dot', line_color=PALETA['neutral'],
    annotation_text='Promedio general del catálogo', annotation_position='bottom right',
    annotation_font_size=10,
)

# Se anotan dinámicamente los dos casos discutidos en la narrativa (Sección 5)
fila_documental = chart5_df[(chart5_df.content_type == 'Película') & (chart5_df.genres == 'Documentary')].iloc[0]
fila_horror = chart5_df[(chart5_df.content_type == 'Película') & (chart5_df.genres == 'Horror')].iloc[0]

fig5.add_annotation(x=fila_documental['popularidad_promedio'], y=fila_documental['calificacion_promedio'],
                     text='Documental: joya de nicho', showarrow=True, arrowhead=2, ax=45, ay=-25,
                     font=dict(size=10, color=PALETA['texto']))
fig5.add_annotation(x=fila_horror['popularidad_promedio'], y=fila_horror['calificacion_promedio'],
                     text='Horror: alto volumen,<br>menor calificación', showarrow=True, arrowhead=2, ax=25, ay=30,
                     font=dict(size=10, color=PALETA['texto']))

fig5.update_layout(
    **LAYOUT_BASE,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    xaxis=dict(showgrid=True, gridcolor='#E5E7EB'),
    yaxis=dict(showgrid=True, gridcolor='#E5E7EB'),
    height=560,
)
fig5.show()

**Justificación teórica — Gráfico 5**

- **Selección según tipo de dato:** combinamos dos variables numéricas continuas (posición) con una tercera numérica (tamaño) y una categórica (color) — el caso de uso de un **gráfico de dispersión de burbujas**. Es la única representación que permite comparar simultáneamente **tres magnitudes por género** (popularidad, calificación, volumen) sin recurrir a una tabla, que exigiría lectura secuencial en vez de comparación visual instantánea.
- **Atributos visuales y preatentivos:** se usan **cuatro atributos preatentivos en capas jerárquicas de precisión**: posición X/Y (los más precisos, para las dos variables centrales del análisis), **tamaño** para volumen (variable de apoyo, no crítica para la decisión) y **color** para tipo de contenido. Ubicar la variable menos crítica (volumen) en el atributo perceptualmente menos preciso (tamaño) es intencional: evita que una variable secundaria distraiga de la comparación principal.
- **Percepción visual y Gestalt:** la línea punteada de promedio general aplica el principio de **conexión visual con referencia**: cualquier burbuja por encima o por debajo se interpreta instantáneamente contra un punto de comparación común, sin que la audiencia deba calcular el promedio mentalmente. Las anotaciones señaladas (Documental, Horror) rompen deliberadamente la uniformidad del resto de las burbujas — un caso de **jerarquía visual por contraste**, guiando la mirada a los dos casos que se retoman en la narrativa de la Sección 5 sin necesidad de etiquetar las 26 burbujas restantes (lo que saturaría el gráfico).
- **Carga cognitiva:** se optó por **etiquetar solo 2 de las 26 burbujas** (revelación progresiva vía *hover* para el resto) en lugar de mostrar todas las etiquetas de forma permanente — un gráfico con 26 etiquetas de texto superpuestas sería ilegible. El título comunica la tensión central del hallazgo (mejor valorado ≠ más popular) sin que la audiencia deba descubrirlo por sí misma explorando el gráfico primero.

## 5. Narrativa visual (Data Storytelling)

> **Para presentar al Comité Directivo de Contenido — StreamView Analytics**

**1. Contexto.**
StreamView Analytics necesita decidir dónde concentrar su inversión de contenido entre Películas y Series, a través de géneros y mercados, para maximizar engagement y retención. Analizamos el catálogo completo: 31.991 títulos (16.000 películas y 15.991 series, tras limpieza) lanzados entre 2010 y 2025, usando popularidad, volumen de votos y calificación de audiencia como *proxies* de engagement y calidad percibida.

**2. Hallazgo principal.**
**Las Series TV superan de forma sistemática a las Películas**, tanto en calificación de audiencia (7,03 vs. 6,31 sobre 10, en títulos con datos de calificación) como en popularidad mediana (más de 3 veces superior). Esta ventaja no es un artefacto de un año o un género puntual: se sostiene durante los 16 años analizados (Gráfico 2) y en 5 de los 8 géneros directamente comparables entre ambos formatos (Gráfico 1), y los 8 géneros mejor evaluados de todo el catálogo integrado son, sin excepción, géneros de Serie (Gráfico 5).

**3. Evidencia.**
- **Gráfico 1** muestra que, en volumen, las Series ya igualan o superan a las Películas en la mayoría de los géneros con taxonomía comparable.
- **Gráfico 2** confirma que la brecha de calificación Serie > Película es estructural, estable a lo largo de 16 años, y no producto de ruido reciente.
- **Gráfico 5** ubica a los 8 géneros de Serie evaluados por sobre el promedio general del catálogo, mientras que Documental (Película) es la única excepción notable del lado de Películas: mejor valorado que cualquier género de Película, aunque con baja popularidad — una "joya de nicho".
- **Gráfico 4** añade el eje geográfico: Japón y China no solo aportan volumen relevante de Series, sino también calificación por sobre el promedio del top 10 de países.
- **Gráfico 3** matiza la conclusión: popularidad y calificación están débilmente correlacionadas (r ≈ 0,16) — por lo tanto, **la recomendación no puede basarse en una sola métrica**.

**4. Conclusión.**
Para StreamView Analytics, el formato Serie constituye actualmente el motor más confiable de engagement y satisfacción de audiencia del catálogo. Sin embargo, el hallazgo del Gráfico 3 advierte que "popular" y "bien evaluado" son señales distintas: optimizar solo por popularidad arriesga sacrificar calidad percibida, y viceversa. El género Horror en Películas (alto volumen, la calificación más baja del catálogo) representa el caso opuesto — posible sobre-inversión en un segmento de baja satisfacción.

**5. Recomendación.**
1. **Priorizar la producción original de Series** en los géneros donde ya existe señal fuerte de calidad y volumen (Animación, Acción & Aventura, Sci-Fi & Fantasía, Comedia, Drama), y en los mercados de Japón, China y Corea del Sur.
2. **Proteger el catálogo de Documentales** en Películas como diferenciador de calidad de nicho, sin exigirle métricas de popularidad masiva.
3. **Auditar la inversión en Horror (Película)**: es el 4° género más grande del catálogo de Películas, pero tiene la calificación más baja de todo el análisis.
4. **Adoptar un KPI compuesto** (no solo popularidad, no solo calificación) para decisiones de luz verde de contenido, dado que ambas métricas están débilmente correlacionadas.
5. **Profundizar este análisis con datos de comportamiento real de usuario** (retención, tiempo de visualización, *churn* por suscriptor) antes de comprometer presupuesto — ver limitaciones en la Sección 6.

## 6. Evaluación crítica de la solución

> **Dashboard interactivo complementario.** La pauta EP1/EP2 exige entregar y mostrar un dashboard interactivo con KPIs, filtros, navegación e interacción — no solo proponerlo. Ese dashboard se entrega como archivo aparte, `dashboard_streamview_analytics.py` (Streamlit), y usa exactamente la misma limpieza de datos y la misma paleta corporativa que este notebook. Incluye: **KPIs dinámicos** (títulos, calificación promedio, popularidad mediana, % Serie TV, género principal, recalculados según el filtro activo), **filtros** (tipo de contenido, año, mínimo de votos acumulados, género), **navegación** por pestañas (Resumen, Géneros, Evolución temporal, Popularidad vs. Calificación, Países, Explorar datos) y **funcionalidades de interacción** (zoom/hover en cada gráfico Plotly, tabla filtrable y exportación a CSV). Se ejecuta con `streamlit run dashboard_streamview_analytics.py` desde la misma carpeta que los dos CSV. Este notebook sigue siendo el informe de análisis narrado; el dashboard es la capa de exploración libre para el Comité.

### Fortalezas

- El catálogo integrado es amplio y **balanceado** (16 años, más de 30.000 títulos, más de 80 idiomas, más de 100 países), lo que da robustez estadística a los patrones encontrados.
- Se usaron **métricas de audiencia reales** (`popularity`, `vote_count`, `vote_average`) en lugar de suposiciones, y cada decisión de limpieza quedó documentada y verificada con código (Secciones 2 y 3), no solo asumida.
- El diseño visual sigue una **paleta y tipografía consistentes** en los 5 gráficos, con jerarquía clara y justificación teórica explícita para cada decisión.
- Los hallazgos son **accionables** y están directamente conectados a decisiones de inversión de contenido, no solo descriptivos.

### Limitaciones de los datos actuales

- Es un dataset **a nivel de catálogo/título**, no de comportamiento de usuario individual: no existen sesiones, tiempo de visualización, *churn* ni datos de suscripción reales. "Engagement" y "retención" se trabajaron como **proxies** (popularidad y volumen de votos), no como medición directa — esto debe explicitarse en cualquier presentación ejecutiva para no sobrevender la certeza del hallazgo.
- Las columnas `duration` (en ambas fuentes) y `rating` (duplicada de `vote_average`) **no fueron utilizables**, lo que impidió analizar la duración real de las películas o el número real de temporadas de las series.
- `date_added` resultó idéntica a `release_year`, por lo que no se pudo diferenciar "fecha de estreno" de "fecha de incorporación al catálogo de streaming".
- `budget` y `revenue` solo existen para Películas y con cerca de dos tercios de los valores en cero (no reportado), por lo que se excluyeron del análisis principal para no construir conclusiones sobre datos mayormente ausentes.
- El catálogo original tiene **exactamente 1.000 títulos por año** en cada fuente — un patrón de muestra curada, no de crecimiento orgánico, que impide analizar la evolución real del tamaño del catálogo en el tiempo.
- Los títulos de **2024–2025** están afectados por sesgo de "arranque en frío" en `vote_count` y `popularity` (se documentó y mitigó, pero limita la comparabilidad total de esos años).
- Las **taxonomías de género difieren** entre el catálogo de Películas y el de Series (TMDB usa categorías distintas para cada uno), lo que restringió a 8 el número de géneros directamente comparables.

### Oportunidades de mejora

- **Desplegar el dashboard** (por ejemplo, en Streamlit Community Cloud o un servidor interno) para que el Comité Directivo lo consulte desde el navegador sin instalar nada localmente, y agregar autenticación básica si va a compartirse fuera del equipo del proyecto.
- Incorporar **datos reales de comportamiento de usuario** desde los sistemas internos de StreamView Analytics (retención, *watch time*, dispositivo, *churn*) para pasar de *proxies* de engagement a medición directa.
- Resolver a nivel de fuente la duplicidad `rating`/`vote_average` y recuperar el dato real de `duration`.
- Unificar la taxonomía de género entre catálogos de Película y Serie para ampliar la comparación más allá de las 8 categorías actuales.

## 7. Conclusiones y recomendaciones finales

El análisis del catálogo integrado de StreamView Analytics (31.991 títulos, 2010–2025) entrega una conclusión clara y accionable: **el formato Serie genera consistentemente mayor engagement y mejor percepción de calidad que el formato Película**, patrón que se sostiene en el tiempo, en géneros comparables y en los principales mercados internacionales del catálogo.

Para la toma de decisiones basada en datos, recomendamos a StreamView Analytics:

1. Reorientar progresivamente el mix de inversión de contenido hacia **Series originales**, priorizando los géneros y mercados (Japón, China, Corea del Sur) donde ya existe evidencia de alto volumen y alta calificación simultáneamente.
2. Mantener el catálogo de **Documentales** en Películas como apuesta de calidad de nicho, sin exigirle métricas de alcance masivo.
3. Revisar la estrategia de inversión en **Horror (Película)**, el segmento de mayor volumen con la calificación más baja del catálogo.
4. No optimizar la cartera de contenido con una sola métrica: **popularidad y calificación son señales débilmente correlacionadas** (r ≈ 0,16) y deben monitorearse de forma conjunta.
5. Tratar estos hallazgos como una **primera capa de evidencia**: el paso natural siguiente es cruzar este análisis de catálogo con datos reales de comportamiento de usuario (retención, tiempo de visualización, *churn*) antes de comprometer presupuesto de producción a gran escala, y usar el dashboard interactivo complementario (Sección 6) para que el Comité Directivo pueda monitorear estos indicadores de forma continua, no solo en el momento de esta presentación.

---
*Fin del notebook — Evaluación Parcial EP1 (Encargo) y EP2 (Presentación), ADY1104 Visualización de Datos.*